In [14]:
import torch # type:ignore
import torch.nn as nn # type:ignore
import torch.optim as optim # type:ignore
import torchvision # type:ignore
import torchvision.transforms as transforms # type:ignore
from torch.utils.data import DataLoader # type:ignore/
from torchvision.datasets import ImageFolder # type:ignore

In [15]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

Using device: cuda


In [16]:
# transforms.Compose([]) -> It groups multiple image transformation steps into a single pipeline. When an image passes through this pipeline, it gets sequentially modified by each operation inside the list.

# transforms.RandomCrop(32, padding=4) -> It pads the original 32*32 pixel image by 4 pixels an all sides (making 40x40) and then randomly crops a 32x32 section out of it.

# transforms.RandomHorizontalFlip() -> Another data augmentation trick. A dog or a car facing left is still a dog or a car when facing right, which helps the model generalize better to new images. Notice this is not applied to transform_test, because we want our evaluation metrics to remain consistent and deterministic.

# transforms.ToTensor() -> Converts a PIL Image or NumPy array (which has pixel values ranging from 0 to 255 with shape [Height, Width, Channel]) into a PyTorch Tensor.It also scales the pixel values down to a floating-point range of [0.0, 1.0] and rearranges the shape to [Channel, Height, Width], which is what PyTorch expects.

transform_train = transforms.Compose([
    transforms.RandomCrop(32, padding=4),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize((0.4914, 0.4822, 0.4465), (0.2023, 0.1994, 0.2010))
])

transform_test = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.4914, 0.4822, 0.4465), (0.2023, 0.1994, 0.2010))
])

classes = ('plane', 'car', 'bird', 'cat', 'deer', 'dog', 'frog', 'horse', 'ship', 'truck')

In [17]:
# Automatically download CIFAR-10 dataset
trainset = torchvision.datasets.CIFAR10(root='./data', train=True, download=True, transform=transform_train)
testset = torchvision.datasets.CIFAR10(root='./data', train=False, download=True, transform=transform_test)
print(f"Training samples: {len(trainset)}")
print(f"Testing samples: {len(testset)}")

Training samples: 50000
Testing samples: 10000


In [18]:
# 1. Create DataLoaders to batch and shuffle the data
trainloader = DataLoader(trainset, batch_size=64, shuffle=True, num_workers=2)
testloader = DataLoader(testset, batch_size=64, shuffle=False, num_workers=2)

In [19]:
# 2. Define a simple Convolutional Neural Network (CNN)
import torch.nn.functional as F

class SimpleCNN(nn.Module):
    def __init__(self):
        super(SimpleCNN, self).__init__()
        self.conv1 = nn.Conv2d(3, 32, 3, padding=1)
        self.conv2 = nn.Conv2d(32, 64, 3, padding=1)
        self.pool = nn.MaxPool2d(2, 2)
        self.fc1 = nn.Linear(64 * 8 * 8, 512)
        self.fc2 = nn.Linear(512, 10)

    def forward(self, x):
        x = self.pool(F.relu(self.conv1(x)))
        x = self.pool(F.relu(self.conv2(x)))
        x = torch.flatten(x, 1)
        x = F.relu(self.fc1(x))
        x = self.fc2(x)
        return x

net = SimpleCNN().to(device)
print(net)

SimpleCNN(
  (conv1): Conv2d(3, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (conv2): Conv2d(32, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (pool): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  (fc1): Linear(in_features=4096, out_features=512, bias=True)
  (fc2): Linear(in_features=512, out_features=10, bias=True)
)


In [ ]:
# 3. Define the Loss function and Optimizer
criterion = nn.CrossEntropyLoss()
# optimizer = optim.SGD(net.parameters(), lr=0.009, momentum=0.7)
# optimizer = optim.RMSprop(net.parameters(), lr=0.0009, alpha=0.9)
optimizer = optim.Adam(net.parameters(), lr=0.00009)

In [38]:
# 4. Training Loop
epochs = 5
for epoch in range(epochs):
    running_loss = 0.0
    for i, data in enumerate(trainloader, 0):
        inputs, labels = data[0].to(device), data[1].to(device)

        # zero the parameter gradients
        optimizer.zero_grad()

        # forward + backward + optimize
        outputs = net(inputs)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        running_loss += loss.item()
        if i % 200 == 199:    # print every 200 mini-batches
            print(f'[{epoch + 1}, {i + 1:5d}] loss: {running_loss / 200:.3f}')
            running_loss = 0.0

print('Finished Training')

[1,   200] loss: 0.479
[1,   400] loss: 0.461
[1,   600] loss: 0.462
[2,   200] loss: 0.456
[2,   400] loss: 0.462
[2,   600] loss: 0.457
[3,   200] loss: 0.444
[3,   400] loss: 0.449
[3,   600] loss: 0.457
[4,   200] loss: 0.438
[4,   400] loss: 0.450
[4,   600] loss: 0.449
[5,   200] loss: 0.440
[5,   400] loss: 0.440
[5,   600] loss: 0.429
Finished Training


In [39]:
# 5. Evaluation Loop on Test Data
correct = 0
total = 0
with torch.no_grad():
    for data in testloader:
        images, labels = data[0].to(device), data[1].to(device)
        outputs = net(images)
        _, predicted = torch.max(outputs.data, 1)
        total += labels.size(0)
        correct += (predicted == labels).sum().item()

print(f'Accuracy of the network on the 10000 test images: {100 * correct / total:.2f} %')

Accuracy of the network on the 10000 test images: 81.51 %
